Scientific source notebook for the isolated reproduction tree. Calculation cells and saved execution objects are preserved. Narrative navigation has been adapted for this package. Use the reproduction runner and frozen-input prerequisites before executing. Historical evaluation blocks remain reused evidence; this copy does not establish independent replication.


# 05 · Downside sensitivity and own momentum

**Question:** How differently does a security respond on negative-market days, and how strong was its own prior trend?

Ordinary and downside regressions use SPY excess returns with intercepts. Downside days have SPY excess return below zero; the main estimate needs 100 such observations and 400 complete observations overall. Momentum is P(t−21)/P(t−252)−1.

Research report · Method and code guide · Evidence map


**Chapter route**

Measurement and interpretation principles · Data and design context · conditional beta and momentum · Read and run this notebook · Calculation source: downside momentum · Results, examples and interpretation · Worked example: momentum is a price ratio, not a beta · Visual evidence · Findings · Analysis and takeaways · Bridge through the study


## Measurement and interpretation principles

Ordinary beta is covariance with SPY excess return divided by its variance. Downside beta applies the corresponding calculation on negative-SPY-excess-return days, subject to its own coverage rules. Own-stock momentum summarizes trailing price performance with the recent month excluded; it is not the French traded momentum factor.


## Data and design context

Ordinary SPY beta and negative-market beta use an intercept and daily excess returns. HC3 standard errors retain uncertainty from each fitted slope, and amplification uncertainty accounts for the overlapping samples. Own momentum is the separate adjusted-price ratio P(t-21)/P(t-252)-1. September 2026 beta estimates stop at August 31; September momentum uses September 30 endpoints. French momentum is a daily factor portfolio with its own convention, distinct from this stock characteristic and from its estimated factor loading. All rows, including ineligible securities, remain visible.


## conditional beta and momentum

`beta_estimates` computes slopes with an intercept; `_slope_fit` uses HC3 influence terms. Downside-amplification uncertainty accounts for overlap between ordinary and downside samples. `momentum_at_formation` uses exact trading-session endpoints and rejects quarantined intervals. This cell rebuilds both windows through the shared loader.


## Read and run this notebook

The calculation definitions now live here, alongside the purpose and interpretation. Use the chapter route for the argument and the bold function names for individual definitions. Definition cells do not download data or run the full study. The results/examples follow the definitions. Save edits and restart the kernel before a full rerun so shared imports read the saved version.


## Calculation source: downside momentum

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [1]:
"""Ordinary and conditional SPY betas, with separately dated own 12-1 momentum.

All regressions include an intercept. HC3 standard errors describe conditional estimation
uncertainty, not uncertainty from selecting a universe or a downside state. The amplification
standard error uses aligned slope influences from both regressions, preserving their dependence.
"""

import sys

import numpy as np
import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.foundation import calendar, eligibility, loader
from quiet_index_500.research_io import save_table


**_slope_fit**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [2]:
def _slope_fit(stock, market):
    centered_market = market - market.mean()
    sum_squared_market = centered_market @ centered_market
    if len(stock) < 3 or sum_squared_market <= np.finfo(float).eps:
        return None
    slope = centered_market @ (stock - stock.mean()) / sum_squared_market
    intercept = stock.mean() - slope * market.mean()
    residuals = stock - intercept - slope * market
    leverage = 1 / len(stock) + centered_market ** 2 / sum_squared_market
    if np.any(1 - leverage <= 1e-12):
        return None
    slope_influence = centered_market * residuals / ((1 - leverage) * sum_squared_market)
    return {"beta": slope, "intercept": intercept,
            "beta_se": np.sqrt(slope_influence @ slope_influence), "influence": slope_influence}


**beta_estimates**

Fit all-day, negative-SPY and worst-decile regressions on matched nonmissing days.

The caller applies the shared window eligibility first. Missing observations in either
input are removed jointly; the downside and decile selections use SPY excess returns.


In [3]:
def beta_estimates(stock_excess, spy_excess, minimum_negative_days):
    """Fit all-day, negative-SPY and worst-decile regressions on matched nonmissing days.

    The caller applies the shared window eligibility first. Missing observations in either
    input are removed jointly; the downside and decile selections use SPY excess returns.
    """
    joint = pd.concat([stock_excess.rename("stock"), spy_excess.rename("spy")], axis=1).dropna()
    stock = joint.stock.to_numpy(dtype=float)
    market = joint.spy.to_numpy(dtype=float)
    result = {"observations": len(joint), "negative_market_observations": int((market < 0).sum()),
              "worst_decile_observations": 0, "ordinary_eligible": False,
              "downside_eligible": False, "worst_decile_eligible": False,
              "ordinary_reason": "degenerate_market", "downside_reason": "insufficient_negative_market_days"}
    for name in ["ordinary", "downside", "worst_decile"]:
        for statistic in ["beta", "intercept", "beta_se"]:
            result[f"{name}_{statistic}"] = np.nan
    result["downside_amplification"] = result["downside_amplification_se"] = np.nan
    if len(joint) < 3:
        return result
    ordinary_fit = _slope_fit(stock, market)
    if ordinary_fit is None:
        return result
    result.update(ordinary_eligible=True, ordinary_reason="eligible")
    for statistic in ["beta", "intercept", "beta_se"]:
        result[f"ordinary_{statistic}"] = ordinary_fit[statistic]
    downside_mask = market < 0
    decile_threshold = np.quantile(market, settings.worst_decile_quantile)
    decile_mask = market <= decile_threshold
    result["worst_decile_observations"] = int(decile_mask.sum())
    result["worst_decile_market_threshold"] = decile_threshold
    for name, mask, minimum in [("downside", downside_mask, minimum_negative_days),
                                ("worst_decile", decile_mask, 3)]:
        if mask.sum() < minimum:
            continue
        fit = _slope_fit(stock[mask], market[mask])
        if fit is None:
            if name == "downside":
                result["downside_reason"] = "degenerate_downside_market"
            continue
        result[f"{name}_eligible"] = True
        for statistic in ["beta", "intercept", "beta_se"]:
            result[f"{name}_{statistic}"] = fit[statistic]
        if name == "downside":
            result["downside_reason"] = "eligible"
            result["downside_amplification"] = fit["beta"] - ordinary_fit["beta"]
            downside_influence = np.zeros(len(joint))
            downside_influence[mask] = fit["influence"]
            difference_influence = downside_influence - ordinary_fit["influence"]
            result["downside_amplification_se"] = np.sqrt(difference_influence @ difference_influence)
    return result


**momentum_at_formation**

Own momentum by ticker from exact session endpoints, retaining all ineligible rows.

At least 200 of the 231 daily returns strictly after the older endpoint and through the
newer endpoint must be valid. A quarantined interval return disqualifies this price ratio:
otherwise an excluded move would re-enter through the two endpoint prices.


In [4]:
def momentum_at_formation(price_panels, daily_returns, formation_date):
    """Own momentum by ticker from exact session endpoints, retaining all ineligible rows.

    At least 200 of the 231 daily returns strictly after the older endpoint and through the
    newer endpoint must be valid. A quarantined interval return disqualifies this price ratio:
    otherwise an excluded move would re-enter through the two endpoint prices.
    """
    prices = price_panels.adjusted_close
    sessions = prices.index
    end_position = sessions.searchsorted(pd.Timestamp(formation_date), side="right") - 1
    older_position = end_position - settings.momentum_start_lag_sessions
    newer_position = end_position - settings.momentum_recent_lag_sessions
    output = pd.DataFrame(index=prices.columns).rename_axis("ticker")
    output["formation_date"] = pd.Timestamp(formation_date)
    output["price_measurement_cutoff"] = sessions[end_position] if end_position >= 0 else pd.NaT
    output["momentum_start_date"] = pd.NaT
    output["momentum_end_date"] = pd.NaT
    output["momentum_observations"] = 0
    output["momentum_quarantined_returns"] = 0
    output["own_momentum"] = np.nan
    output["momentum_eligible"] = False
    output["momentum_reason"] = "insufficient_endpoint_history"
    if older_position < 0:
        return output.reset_index()
    older_date, newer_date = sessions[older_position], sessions[newer_position]
    interval = sessions[older_position + 1:newer_position + 1]
    history = sessions[older_position:end_position + 1]
    old_price, new_price = prices.loc[older_date], prices.loc[newer_date]
    quarantined = price_panels.quarantined_returns.reindex(interval).sum()
    valid_returns = daily_returns.reindex(interval).mask(price_panels.quarantined_returns.reindex(interval)).notna().sum()
    before_start = price_panels.before_same_security_start.reindex(history).any()
    endpoint_valid = old_price.gt(0) & new_price.gt(0)
    reason = np.select([before_start, ~endpoint_valid, quarantined.gt(0),
                        valid_returns.lt(settings.momentum_minimum_valid_returns_between_endpoints)],
                       ["before_same_security_start", "missing_endpoint", "quarantined_interval_return",
                        "insufficient_intervening_returns"], default="eligible")
    output["momentum_start_date"] = older_date
    output["momentum_end_date"] = newer_date
    output["momentum_observations"] = valid_returns
    output["momentum_quarantined_returns"] = quarantined
    output["momentum_reason"] = reason
    output["momentum_eligible"] = output.momentum_reason.eq("eligible")
    output["own_momentum"] = (new_price / old_price - 1).where(output.momentum_eligible)
    return output.reset_index()


**build_downside_momentum_tables**

Build security-level characteristics for every formation and both estimation windows.


In [5]:
def build_downside_momentum_tables(apply_corrections=True):
    """Build security-level characteristics for every formation and both estimation windows."""
    panels = loader.load_price_panels(apply_corrections=apply_corrections)
    returns = loader.load_daily_returns(apply_corrections=apply_corrections)
    excess = loader.load_excess_returns(apply_corrections=apply_corrections)
    spy = loader.load_spy_excess_returns(apply_corrections=apply_corrections)
    factor_cutoff = loader.factor_data_last_date()
    formations = calendar.formation_dates(returns.index)
    windows = [(settings.main_window_sessions, settings.main_window_minimum_observations,
                settings.downside_minimum_negative_market_days_main),
               (settings.sensitivity_window_sessions, settings.sensitivity_window_minimum_observations,
                settings.downside_minimum_negative_market_days_sensitivity)]
    beta_rows, momentum_tables = [], []
    for formation in formations:
        cutoff = calendar.measurement_cutoff(formation, factor_cutoff)
        momentum_tables.append(momentum_at_formation(panels, returns, formation))
        for window_sessions, minimum_observations, minimum_negative in windows:
            window = calendar.estimation_window(returns.index, cutoff, window_sessions)
            eligible = eligibility.panel_window_eligibility(excess, window, minimum_observations, spy,
                                                            panels.before_same_security_start, panels.quarantined_returns)
            stock_in_window, spy_in_window = excess.reindex(window), spy.reindex(window)
            negative_counts = stock_in_window.notna().mul(spy_in_window.lt(0), axis=0).sum()
            for ticker, status in eligible.iterrows():
                row = {"ticker": ticker, "formation_date": formation, "window_sessions": window_sessions,
                       "factor_measurement_cutoff": cutoff, "estimation_first_session": window.min(),
                       "estimation_last_session": window.max(), "quarantined_in_window": status.quarantined_in_window,
                       "observations": status.observations, "ordinary_eligible": False, "downside_eligible": False,
                       "worst_decile_eligible": False, "ordinary_reason": status.reason,
                       "downside_reason": status.reason, "negative_market_observations": negative_counts[ticker],
                       "worst_decile_observations": 0}
                if status.eligible:
                    row.update(beta_estimates(stock_in_window[ticker], spy_in_window, minimum_negative))
                beta_rows.append(row)
    beta_table = pd.DataFrame(beta_rows)
    momentum_table = pd.concat(momentum_tables, ignore_index=True)
    characteristics = beta_table.merge(momentum_table, on=["ticker", "formation_date"], validate="many_to_one")
    coverage = characteristics.groupby(["formation_date", "window_sessions"], as_index=False).agg(
        securities=("ticker", "size"), ordinary_eligible=("ordinary_eligible", "sum"),
        downside_eligible=("downside_eligible", "sum"), momentum_eligible=("momentum_eligible", "sum"),
        median_ordinary_beta=("ordinary_beta", "median"), median_downside_beta=("downside_beta", "median"),
        median_downside_amplification=("downside_amplification", "median"),
        median_own_momentum=("own_momentum", "median"))
    return {"downside_momentum": characteristics, "own_momentum": momentum_table, "coverage": coverage}


**write_downside_momentum_outputs**

Write characteristics, own momentum and coverage with the complete shared provenance.


In [6]:
def write_downside_momentum_outputs(tables=None):
    """Write characteristics, own momentum and coverage with the complete shared provenance."""
    tables = build_downside_momentum_tables() if tables is None else tables
    return {name: save_table(table, "downside_momentum", f"{name}.csv", source_modules=[settings.repository_root_folder / 'src/quiet_index_500/downside_momentum.py'],
                             note="Betas use SPY excess returns and HC3 standard errors; own momentum uses separate formation-date prices.")
            for name, table in tables.items()}


## Results, examples and interpretation

The cells below use the definitions above and the frozen study inputs.


In [7]:
from quiet_index_500 import downside_momentum, settings
tables = build_downside_momentum_tables()
paths = write_downside_momentum_outputs(tables)
{name: path.relative_to(settings.repository_root_folder).as_posix() for name, path in paths.items()}

{'downside_momentum': 'results/downside_momentum/downside_momentum.csv',
 'own_momentum': 'results/downside_momentum/own_momentum.csv',
 'coverage': 'results/downside_momentum/coverage.csv'}

In [8]:
tables['coverage']

,formation_date,window_sessions,securities,ordinary_eligible,downside_eligible,momentum_eligible,median_ordinary_beta,median_downside_beta,median_downside_amplification,median_own_momentum
0,2017-12-29,252,503,466,466,465,1.024295,1.015460,-0.025493,0.218670
1,2017-12-29,504,503,461,461,465,1.071355,1.075797,0.009609,0.218670
2,2018-03-29,252,503,468,468,466,0.922704,0.913690,-0.003518,0.195056
3,2018-03-29,504,503,463,463,466,1.005818,0.984149,-0.008716,0.195056
4,2018-06-29,252,503,468,468,468,0.922411,0.934690,0.017709,0.141304
...,...,...,...,...,...,...,...,...,...,...
67,2026-03-31,504,503,497,497,499,0.787845,0.854674,0.033153,0.177112
68,2026-06-30,252,503,499,499,499,0.657341,0.650547,-0.001423,0.115227
69,2026-06-30,504,503,497,497,499,0.756076,0.831860,0.037952,0.115227
70,2026-09-30,252,503,501,501,500,0.570246,0.669806,0.066154,0.109578


In [9]:
tables['downside_momentum'].tail(10)

,ticker,formation_date,window_sessions,factor_measurement_cutoff,estimation_first_session,estimation_last_session,quarantined_in_window,observations,ordinary_eligible,downside_eligible,...,downside_amplification_se,worst_decile_market_threshold,price_measurement_cutoff,momentum_start_date,momentum_end_date,momentum_observations,momentum_quarantined_returns,own_momentum,momentum_eligible,momentum_reason
36206,NCLH,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.431894,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,-0.347879,True,eligible
36207,AOS,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.269408,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,-0.169095,True,eligible
36208,DVA,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.413997,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,0.347806,True,eligible
36209,HRL,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.211499,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,-0.066700,True,eligible
36210,WYNN,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.321617,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,-0.304276,True,eligible
36211,ERIE,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.306893,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,-0.166883,True,eligible
36212,PSKY,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.390045,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,-0.432756,True,eligible
36213,MGM,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.322199,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,0.132220,True,eligible
36214,BF-B,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.444349,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,0.007056,True,eligible
36215,NWS,2026-09-30,252,2026-08-31,2025-08-29,2026-08-31,0,252,True,True,...,0.259306,-0.010039,2026-09-30,2025-09-29,2026-08-31,231,0,0.002231,True,eligible


## Worked example: momentum is a price ratio, not a beta

Illustrative inputs are separate from study data. Re-run this short cell to connect the method to its numerical output.


In [10]:
# Hypothetical adjusted prices at the exact trading-session endpoints.
older_price, recent_excluded_month_endpoint = 100., 120.
own_momentum_example = recent_excluded_month_endpoint / older_price - 1
ordinary_beta_example, downside_beta_example = 1.1, 1.4
{'own_momentum': round(own_momentum_example, 4),
 'downside_amplification': round(downside_beta_example - ordinary_beta_example, 4)}

{'own_momentum': 0.2, 'downside_amplification': 0.3}

## Visual evidence

Source figure: 03_characteristic_overlap

These figures use saved results; see the figure guide for sources, sample definitions and limitations.


## Findings

Ordinary and downside beta have a mean within-formation rank correlation of 0.932 in the company panel. They overlap substantially. Momentum enters forward controls, but downside beta is descriptive and is not a regressor in the implemented forward tests.

## Analysis and takeaways

Ordinary and downside beta have a formation-averaged Spearman correlation of 0.932. Conditional downside sensitivity is descriptive here and is not an added regressor in Notebook 08. Neither beta nor momentum alone establishes a trading strategy or incremental forward-risk information.

## Bridge through the study

07 combines these measurements with factors, size and liquidity. 08 uses own momentum as a control. This notebook does not independently demonstrate incremental predictive value for downside sensitivity.
